In [71]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

from pathlib import Path
import sys
sys.path.append(str(Path('../python').resolve()))

from run_youtube_pipeline import extract_pipeline, clean_yt_data

# Define Client

In [2]:
client = bigquery.Client()  # uses gcloud credentials automatically

## Clean Data

In [ ]:
# user_name = "TaylorSwift"
# max_videos=5
# info_mv = extract_pipeline(user_name, max_videos)
# df = pd.DataFrame(info_mv)
# df_clean = clean_yt_data(df.copy())
# df.description.iloc[0]
# df_clean.description.iloc[0]
# df_clean.head(1)
# if len(info_mv) != df.shape[0]:
#     Exception("Not matching entries from list -> df")
# # df.info()
# #pd.isna(df.description.iloc[0])
# #df_mv[df_mv.isna().any(axis=1)]
# # Replace NaN, empty strings, or "NULL" (case-insensitive) with "None"
# df['description'] = df['description'].apply(
#     lambda x: "None" if (pd.isna(x) or str(x).strip() == "" or str(x).strip().upper() == "NULL") else x
# )

# #missing values aren’t converted to "nan" but instead to "None"
# df = df.fillna("None").astype(str)
# df = df.astype(str)
# df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# import numpy as np
# # Replace string "None" with real NaN
# df['published'] = df['published'].replace("None", np.nan)

# # Convert to datetime, invalid values will become NaT
# df['published'] = pd.to_datetime(df['published'], errors='coerce')
# df
# df['published'] = pd.to_datetime(df[~df['published'].isna()][['published']])
# for i in range(len(info_mv)):
#     info = info_mv[i]
    


# Read Tables

In [3]:
def read_bq_table(project_id, dataset_id, table_id, max_records=10):
    
    query = f"SELECT * FROM `{project_id}.{dataset_id}.{table_id}` LIMIT {max_records}"
    df = client.query(query).to_dataframe()
    return df


In [4]:
project_id = "swiftie-friend" 
dataset_id = "album_songs_summary"
table_id = "album_songs_summary"
max_records = 10
df = read_bq_table(project_id, dataset_id, table_id, max_records)
df.head(1)

/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,track_name,track_id,duration_ms,explicit,track_number,album_id,album_name,album_release_date,album_total_tracks,album_type,artist_name,artist_id,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,imgonnagetyouback,1kcwpPDQnqEqmezzXdJTCP,222072,False,18,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.784,0.391,-9.471,0.0633,0.608,0.0,0.252,0.15


# Create New Table
Create first the dataset on UI if doesnt exist

## Load data by UPLOADING local file

In [5]:
project_id = "swiftie-friend" 
dataset_id = "social_media"
table_id = "youtube_music_videos"
table_ref = f"{project_id}.{dataset_id}.{table_id}"


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # skip header row
    autodetect=True,      # let BigQuery detect schema
)

input_path="../data/"
input_file_name="youtube_taylor_last_2_mv"
#df = pd.read_csv(f"{input_path}{input_file_name}.csv")

with open(f"{input_path}{input_file_name}.csv", "rb") as source_file:
    job = client.load_table_from_file(source_file, table_ref, job_config=job_config)
    job.result()  # Wait for completion



In [6]:
read_bq_table(project_id, dataset_id, table_id, max_records)

/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,etag,id,title,published,description,link,thumbnails
0,fpHY3xk0WzFvihSqUggnRLthxvc,w_DD-7_zVtw,"And, baby, that’s show business for you. New a...",2025-08-14 05:00:58+00:00,None,https://www.youtube.com/watch?v=w_DD-7_zVtw,https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg
1,laHqQ7cAS7ySrB0qZm2Tnnh6ZNE,P0haCYjysUs,Anti-Hero (Behind The Scenes with The Ghosts I...,2024-12-13 17:24:03+00:00,"Behind the scenes of ""Anti-Hero"" official musi...",https://www.youtube.com/watch?v=P0haCYjysUs,https://i.ytimg.com/vi/P0haCYjysUs/hqdefault.jpg


## Load data, Define Table and its Schema

In [8]:
# project_id = "swiftie-friend" 
# dataset_id = "social_media"
# table_id = "youtube_music_videos"
# max_records = 10
# df = read_bq_table(project_id, dataset_id, table_id, max_records)
# df.head(1)

In [ ]:
input_path="../data/yt_musics_videos/"
input_file_name="youtube_taylor_last_5_mv"
#Pandas doesn’t preserve dtypes in CSV (CSV is plain text).
# It serializes your datetime values into ISO-like strings, e.g. "2025-08-21 12:34:56".
# When you open the .csv in Excel/Notepad or reload it with pd.read_csv, it comes back as object (string) unless you explicitly parse it.
df = pd.read_csv(f"{input_path}{input_file_name}.csv", parse_dates=["published"])


In [28]:
df.dtypes

etag                        object
id                          object
title                       object
published      datetime64[ns, UTC]
description                 object
link                        object
thumbnails                  object
dtype: object

In [30]:
project_id = "swiftie-friend" 
dataset_id = "social_media"
table_id = "youtube_music_videos_v2"
table_ref = f"{project_id}.{dataset_id}.{table_id}"

#TIMESTAMP in BigQuery = an absolute point in time (stored internally as UTC).
#Pandas column datetime64[ns, UTC] is timezone-aware and maps perfectly to BigQuery TIMESTAMP.

schema = [
    bigquery.SchemaField("etag", "STRING"),
    bigquery.SchemaField("id", "STRING"),
    bigquery.SchemaField("title", "STRING"),
    bigquery.SchemaField("published", "TIMESTAMP"),
    bigquery.SchemaField("description", "STRING"),
    bigquery.SchemaField("link", "STRING"),
    bigquery.SchemaField("thumbnails", "STRING"),
]

table = bigquery.Table(table_ref, schema=schema)
table = client.create_table(table, exists_ok=True)  # overwrite if exists
print(f"Table {table.table_id} created with schema.")

Table youtube_music_videos_v2 created with schema.


In [32]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # skip header row
    autodetect=False,      # Dont BigQuery detect schema
)


In [33]:
with open(f"{input_path}{input_file_name}.csv", "rb") as source_file:
    job = client.load_table_from_file(source_file, table_ref, job_config=job_config)


In [194]:
conf = {
    "source_format": bigquery.SourceFormat.CSV,
    "skip_leading_rows": 1,   # skip header row
    "autodetect": False,       # don't let BigQuery detect schema
}

test = bigquery.LoadJobConfig(**conf)

In [34]:
job.result()  # Wait for completion

LoadJob<project=swiftie-friend, location=US, id=9c326efd-6859-494d-9f3b-4fadbf866dbd>

In [36]:
x = read_bq_table(project_id, dataset_id, table_id, max_records)

/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [37]:
x.dtypes

etag                        object
id                          object
title                       object
published      datetime64[us, UTC]
description                 object
link                        object
thumbnails                  object
dtype: object

In [39]:
x

,etag,id,title,published,description,link,thumbnails
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,No Description Provided.,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg
1,fpHY3xk0WzFvihSqUggnRLthxvc,w_DD-7_zVtw,"And, baby, that’s show business for you. New a...",2025-08-14 05:00:58+00:00,No Description Provided.,https://www.youtube.com/watch?v=w_DD-7_zVtw,https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg
2,laHqQ7cAS7ySrB0qZm2Tnnh6ZNE,P0haCYjysUs,Anti-Hero (Behind The Scenes with The Ghosts I...,2024-12-13 17:24:03+00:00,"Behind the scenes of ""Anti-Hero"" official musi...",https://www.youtube.com/watch?v=P0haCYjysUs,https://i.ytimg.com/vi/P0haCYjysUs/hqdefault.jpg
3,feRHhpOZd0x9MRM4mwHCaEbNvfA,PQp643val70,cardigan (behind the scenes - forest &amp; ocean),2024-12-13 15:26:29+00:00,"Behind the scenes of ""cardigan"" official music...",https://www.youtube.com/watch?v=PQp643val70,https://i.ytimg.com/vi/PQp643val70/hqdefault.jpg
4,uGOjeZyN-N4bLk0vbng_Qsn6Tjk,K-8dOw7yuPo,Bejeweled (Behind the Scenes with Laura Dern),2024-12-13 15:03:35+00:00,"Behind the scenes of ""Bejeweled"" official musi...",https://www.youtube.com/watch?v=K-8dOw7yuPo,https://i.ytimg.com/vi/K-8dOw7yuPo/hqdefault.jpg


## Load New Data

In [84]:
input_path="../data/yt_musics_videos/"
input_file_name="youtube_taylor_last_10_mv"
df_new = pd.read_csv(f"{input_path}{input_file_name}.csv", parse_dates=["published"])


In [77]:
new_etags = df.etag.unique()

In [74]:
project_id = "swiftie-friend" 
dataset_id = "social_media"
table_id = "youtube_music_videos_v2"
table_name = f"{project_id}.{dataset_id}.{table_id}"

In [60]:
query = f"SELECT etag  FROM `{project_id}.{dataset_id}.{table_id}`"
query_job = client.query(query)

In [65]:
etag_list = [row.etag for row in query_job]

In [75]:
rows = client.list_rows(table_name, selected_fields=[bigquery.SchemaField("etag", "STRING")])

etag_list = [row[0] for row in rows]  # since only one column selected
print(etag_list)

['jwpriAsHeuWK7mrpU68EaznMp3Y', 'fpHY3xk0WzFvihSqUggnRLthxvc', 'laHqQ7cAS7ySrB0qZm2Tnnh6ZNE', 'feRHhpOZd0x9MRM4mwHCaEbNvfA', 'uGOjeZyN-N4bLk0vbng_Qsn6Tjk']


In [63]:
for q in query_job:
    print(q.etag)

jwpriAsHeuWK7mrpU68EaznMp3Y
fpHY3xk0WzFvihSqUggnRLthxvc
laHqQ7cAS7ySrB0qZm2Tnnh6ZNE
feRHhpOZd0x9MRM4mwHCaEbNvfA
uGOjeZyN-N4bLk0vbng_Qsn6Tjk


In [79]:
query_insert = f"""SELECT etag  FROM `{project_id}.{dataset_id}.{table_id}`
WHERE etag NOT IN {new_etags}"""

In [80]:
query_job = client.query(query)

In [82]:
etag_list = [row.etag for row in query_job]

In [83]:
etag_list

['jwpriAsHeuWK7mrpU68EaznMp3Y',
 'fpHY3xk0WzFvihSqUggnRLthxvc',
 'laHqQ7cAS7ySrB0qZm2Tnnh6ZNE',
 'feRHhpOZd0x9MRM4mwHCaEbNvfA',
 'uGOjeZyN-N4bLk0vbng_Qsn6Tjk']

### 🔹 1. load_table_from_dataframe (bulk load, preferred)

In [86]:
table_name

'swiftie-friend.social_media.youtube_music_videos_v2'

In [88]:
job_config_appending = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"   # or "WRITE_TRUNCATE" to replace everything
)

job = client.load_table_from_dataframe(df_new, table_name, job_config=job_config_appending)
job.result()  


/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=swiftie-friend, location=US, id=e3940d58-7eb3-4301-8b0b-6c68511e0ae6>

### Use MERGE for upsert (insert + update) Only insert rows with new etag

If you want add or update depending on a condition, you can:

Load df_new into a temporary/staging table.

Run a MERGE SQL statement to upsert into your target table.

In [ ]:
input_path="../data/yt_musics_videos/"
input_file_name="youtube_taylor_last_50_mv"
df_new = pd.read_csv(f"{input_path}{input_file_name}.csv", parse_dates=["published"])
df_new.shape

In [100]:
project_id = "swiftie-friend" 
dataset_id = "social_media"
table_id_new = "new_mv"
table_name_new = f"{project_id}.{dataset_id}.{table_id_new}"  # temp dataset/table

main_table= "youtube_music_videos_v2"
main_table_name = f"{project_id}.{dataset_id}.{main_table}" 


In [ ]:
main_table = client.get_table(main_table_name)
print(
    "Loaded {} rows and {} columns to {}".format(
        main_table.num_rows, len(main_table.schema), main_table
    )
)

In [102]:
# schema = [
#     bigquery.SchemaField("etag", "STRING"),
#     bigquery.SchemaField("id", "STRING"),
#     bigquery.SchemaField("title", "STRING"),
#     bigquery.SchemaField("published", "TIMESTAMP"),
#     bigquery.SchemaField("description", "STRING"),
#     bigquery.SchemaField("link", "STRING"),
#     bigquery.SchemaField("thumbnails", "STRING"),
# ]

main_table_schema = main_table.schema

table_new = bigquery.Table(table_name_new, schema=schema)
table_new = client.create_table(table_new, exists_ok=True)  # overwrite if exists
print(f"Table {table_new.table_id} created with schema.")

Table new_mv created with schema.


In [111]:
# Step 1: Load the new DataFrame into a staging table (overwrite each time)
job = client.load_table_from_dataframe(
    df_new,
    table_new,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()


/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=swiftie-friend, location=US, id=e7d48b9e-cf14-49cd-9973-99778644b6b7>

In [105]:
columns = [field.name for field in main_table.schema]

insert_cols = ", ".join(columns)
values = ", ".join([f"S.{col}" for col in columns])


In [112]:
# Step 2: Insert only rows whose etag is not present in the main table
#In BigQuery, column names must be specified in MERGE statements
merge_sql = f"""
MERGE `{main_table_name}` T
USING `{table_new}` S
ON T.etag = S.etag
WHEN NOT MATCHED THEN
  INSERT ({insert_cols})
  VALUES ({values})
"""

query_job = client.query(merge_sql)
query_job.result()
print("Inserted only new rows based on etag")


Inserted only new rows based on etag


In [114]:
stats = query_job._properties["statistics"]["query"]

print("Inserted:", stats.get("numDmlInsertedRowCount", 0))
print("Updated:", stats.get("numDmlUpdatedRowCount", 0))
print("Deleted:", stats.get("numDmlDeletedRowCount", 0))


Inserted: 0
Updated: 0
Deleted: 0


In [ ]:
table = client.get_table(table_id)  # Make an API request.
print(
    "Loaded {} rows and {} columns to {}".format(
        table.num_rows, len(table.schema), table_id
    )
)

In [128]:
df_v2 = df_new.copy()

In [129]:
df_v2 = pd.concat([df_v2, df_new], axis=0)

In [130]:
df_v2.shape

(100, 7)

In [126]:
df_v2.drop_duplicates(['etag'], keep='first', inplace=True)

In [127]:
df_v2.shape

(50, 7)

In [ ]:
titles_duplicated_with_no_description = df_v2[(df_v2.duplicated(['title'])) & (df_v2['description'] == "No Description Provided.")].title.unique()
titles_duplicated_with_description = df_v2[(df_v2.title.isin(titles_duplicated_with_no_description)) & (df_v2['description'] != "No Description Provided.")].title.unique()
#df_v2[df_v2.title.isin(titles_duplicated_with_description) & (df_v2['description'] == "No Description Provided.")]
indexes_to_drop = df_v2[df_v2.title.isin(titles_duplicated_with_description) & (df_v2['description'] == "No Description Provided.")].index
df_v2.drop(index=indexes_to_drop)

In [143]:
titles_with_no_description = df_v2[(df_v2.duplicated(['title'])) & (df_v2['description'] == "No Description Provided.")].title.unique()

In [158]:
titles_duplicated_with_description = df_v2[(df_v2.title.isin(titles_with_no_description)) & (df_v2['description'] != "No Description Provided.")].title.unique()

In [174]:
df_v2[df_v2.title.isin(titles_duplicated_with_description) & (df_v2['description'] == "No Description Provided.")]

,etag,id,title,published,description,link,thumbnails
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,No Description Provided.,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg


In [176]:
indexes_to_drop = df_v2[df_v2.title.isin(titles_duplicated_with_description) & (df_v2['description'] == "No Description Provided.")].index

In [179]:
df_v2[df_v2.etag == "jwpriAsHeuWK7mrpU68EaznMp3Y"]

,etag,id,title,published,description,link,thumbnails
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,pippo,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,No Description Provided.,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg


In [192]:
df_v2[df_v2.etag == "jwpriAsHeuWK7mrpU68EaznMp3Y"].index

Index([0, 0], dtype='int64')

In [185]:
df_v2[df_v2.title.isin(titles_duplicated_with_description) & (df_v2['description'] == "No Description Provided.")]

,etag,id,title,published,description,link,thumbnails
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,No Description Provided.,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg


Index([0], dtype='int64')

In [181]:
df_v3 = df_v2.drop(index=indexes_to_drop)

In [186]:
df_v2[df_v2.etag == "jwpriAsHeuWK7mrpU68EaznMp3Y"]

,etag,id,title,published,description,link,thumbnails
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,pippo,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg
0,jwpriAsHeuWK7mrpU68EaznMp3Y,zqaLqctWP28,The Life of a Showgirl: The Tiny Bubbles in Ch...,2025-08-25 20:02:56+00:00,No Description Provided.,https://www.youtube.com/watch?v=zqaLqctWP28,https://i.ytimg.com/vi/zqaLqctWP28/hqdefault.jpg


In [159]:
titles_duplicated_with_description

array(['The Life of a Showgirl: The Tiny Bubbles in Champagne Vinyl Collection ❤️\u200d🔥'],
      dtype=object)

In [ ]:
df_v2[(df_v2.duplicated(['title'])) & (df_v2['description'] != "No Description Provided.")].

(43, 7)

In [195]:
from load_youtube_videos_info import LoadToBigQuery

In [196]:
load_bq = LoadToBigQuery(project_id, dataset_id, schema, job_config_appending)

AttributeError: 'LoadToBigQuery' object has no attribute 'job_config'